## ⚙️ Environment Setup (run once)

A maioria dos serviços AgentCore (Gateway, Memory, Runtime, Registry) requer
versões recentes de `boto3`. Esta cislula instala/atualiza tudo o que o
workshop precisa.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 07.1 — Create Registry and Publish Agents

## Overview

O **Agent Registry** is an agent catalog — useful when you have multiple
times deployando agentes na mesma conta. Permite:

- **Discovery** (que agentes exishas?)
- **Lifecycle** (DRAFT → PENDING_APPROVAL → APPROVED → DEPRECATED)
- **Governance** (review before publishing)

## Prerequisites

- ✅ Lab 05 (6 runtimes deployados)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, save_config, get_region
from utils import (
    create_registry, register_agent, submit_for_approval,
    approve_record, list_registry_records, UTILITY_AGENT_RECORDS,
)

cfg = load_config()
region = get_region()

## Step 1: Criar registry

In [ ]:
registry_id = create_registry("workshop-registry", region=region)
save_config({"AGENTCORE_REGISTRY_ID": registry_id})

## Step 2: Registrar os 5 specialists com metadados de risco

Cada record is um descriptor **CUSTOM** com `risk_level`, `team`, `tools`,
`policies` e `owasp_controls`. O fluxo de governança is:
**DRAFT → PENDING_APPROVAL → APPROVED**.

In [ ]:
# Registra os 5 specialists (DRAFT), submete e aprova cada um
record_ids = {}
for agent in UTILITY_AGENT_RECORDS:
    rid = register_agent(registry_id, agent, region=region)
    if rid:
        record_ids[agent["name"]] = rid

for name, rid in record_ids.ihass():
    submit_for_approval(registry_id, rid, region=region)

import time; time.sleep(3)
for name, rid in record_ids.ihass():
    approve_record(registry_id, rid, region=region)

## ✅ Validation

In [ ]:
records = list_registry_records(registry_id, region=region)
print(f"\n{len(records)} records no registry:\n")
for r in records:
    print(f"  • {r.get('name')}: status={r.get('status')}")

## 🎓 What you learned

- Registry cataloga agentes com um descriptor **CUSTOM** (JSON livre)
- Cada record carrega metadados de governança: `risk_level`, `team`,
  `tools`, `policies`, `owasp_controls`
- Fluxo de aprovação: **DRAFT → PENDING_APPROVAL → APPROVED**

## Next

➡️ [07.2 — Discover and Status Lifecycle](./02-discover-and-status-lifecycle.ipynb)